In [1]:
import os
def _set_env_from_file(var: str, file_path: str = r"C:\Users\gyanr\gyan-python-workspace\grk_langchain\app\notebooks\data\openai_key.txt"):
    """
    Reads an API key from a specified file and sets it as an environment variable.
    """
    if not os.environ.get(var):
        try:
            # The 'with open' statement ensures the file is closed automatically
            with open(file_path, 'r') as f:
                # Read the first line and strip any leading/trailing whitespace
                key = f.readline().strip()

            if key:
                os.environ[var] = key
                print(f"Successfully loaded {var} from {file_path}")
            else:
                print(f"Warning: {file_path} is empty.")

        except FileNotFoundError:
            print(f"Error: Key file not found at {file_path}. Please create the file.")



In [2]:
_set_env_from_file('OPENAI_API_KEY')

Successfully loaded OPENAI_API_KEY from C:\Users\gyanr\gyan-python-workspace\grk_langchain\app\notebooks\data\openai_key.txt


In [6]:
MODEL="text-embedding-3-small"
#https://github.com/NirDiamant/RAG_Techniques/blob/main/all_rag_techniques/simple_rag_with_llamaindex.ipynb

In [7]:
#%pip install llama-index llama-index-vector-stores-faiss faiss-cpu

In [8]:
from typing import List
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.schema import BaseNode, TransformComponent
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
import faiss
import os
import sys

In [9]:
# Original path append replaced for Colab compatibility

EMBED_DIMENSION = 512

# Chunk settings are way different than langchain examples
# Beacuse for the chunk length langchain uses length of the string,
# while llamaindex uses length of the tokens
CHUNK_SIZE = 200
CHUNK_OVERLAP = 50


# Set embeddig model on LlamaIndex global settings
Settings.embed_model = OpenAIEmbedding(model=MODEL, dimensions=EMBED_DIMENSION)

In [10]:


path = "./data/"
node_parser = SimpleDirectoryReader(input_dir=path, required_exts=['.txt'])
documents = node_parser.load_data()



In [11]:
len(documents)

2

In [12]:
# Create FaisVectorStore to store embeddings
faiss_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=faiss_index)

In [13]:
# class TextCleaner(TransformComponent):
#     """
#     Transformation to be used within the ingestion pipeline.
#     Cleans clutters from texts.
#     """
#     def __call__(self, nodes: List[BaseNode], **kwargs) -> List[BaseNode]:
#         for node in nodes:
#         # get the current content
#             content = node.get_content()
#             # perform your cleaning logic
#             content = content.replace('\t', ' ')
#             content = content.replace(' \n', ' ')
#             # update the node using set_content
#             node.set_content(content)
#             
#             return nodes
# 

from llama_index.core.schema import TransformComponent, BaseNode
from typing import List
import re

class TextCleaner(TransformComponent):
    def __call__(self, nodes: List[BaseNode], **kwargs) -> List[BaseNode]:
        cleaned_nodes = []
        for node in nodes:
            content = node.get_content()
            # 1. basic cleaning
            content = content.replace('\t', ' ').replace(' \n', ' ')
            
            # 2. filter out coordinate noise
            # this regex checks if the content is mostly large groups of numbers/decimals
            numeric_density = len(re.findall(r'\d+\.\d+', content))
            
            # if the node is just a string of coordinates, we skip it
            if numeric_density > 10:
                continue
            
            node.set_content(content)
            cleaned_nodes.append(node)
        
        return cleaned_nodes        

In [14]:
text_splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

# Create a pipeline with defined document transformations and vectorstore
pipeline = IngestionPipeline(
    transformations=[
        TextCleaner(),
        text_splitter,
    ],
    vector_store=vector_store,
)

In [15]:
# Run pipeline and get generated nodes from the process
nodes = pipeline.run(documents=documents)

In [16]:
vector_store_index = VectorStoreIndex(nodes)
retriever = vector_store_index.as_retriever(similarity_top_k=2)

In [17]:
def show_context(context):
    """
    Display the contents of the provided context list.

    Args:
        context (list): A list of context items to be displayed.

    Prints each context item in the list with a heading indicating its position.
    """
    for i, c in enumerate(context):
        print(f"Context {i+1}:")
        print(c.get_content(metadata_mode="none"))
        print(c.text)
        print("\n")

In [18]:


test_query = "What are some of the symptoms of ADHD?"
context = retriever.retrieve(test_query)
show_context(context)



Context 1:
h. Is often easily distracted by extraneous stimuli (for older adolescents and
adults, may include unrelated thoughts).
i. Is often forgetful in daily activities (e.g., doing chores, running errands; for
older adolescents and adults, returning calls, paying bills, keeping
appointments).
2. Hyperactivity and impulsivity: Six (or more) of the following symptoms
have persisted for at least 6 months to a degree that is inconsistent with
developmental level and that negatively impacts directly on social and
academic/occupational activities:
Note: The symptoms are not solely a manifestation of oppositional behavior,
defiance, hostility, or a failure to understand tasks or instructions.
h. Is often easily distracted by extraneous stimuli (for older adolescents and
adults, may include unrelated thoughts).
i. Is often forgetful in daily activities (e.g., doing chores, running errands; for
older adolescents and adults, returning calls, paying bills, keeping
appointments).
2. Hyperacti

### Do not Run again, as the data is already generate

In [5]:
import json
from pathlib import Path
from openai import OpenAI

client = OpenAI()

CLEANED_DIR = Path("./validate/")
OUTPUT_FILE = Path("./validate/q_a.json")
QUESTIONS_PER_FILE = 3

def generate_qa_from_chunk(text: str, section_name: str) -> list[dict]:
    prompt = f"""You are a mental health educator. Based strictly on the following DSM-5 text, 
generate {QUESTIONS_PER_FILE} question and answer pairs.

Rules:
- Questions must be answerable solely from the provided text
- Answers must be factual, concise, and non-diagnostic
- Do not add information not present in the text
- Return only a JSON array like: [{{"question": "...", "answer": "..."}}]

Section: {section_name}
Text:
{text}
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)

    # GPT-4o sometimes wraps the array in a key
    if isinstance(parsed, dict):
        parsed = next(iter(parsed.values()))

    if isinstance(parsed, list) and all(isinstance(item, dict) for item in parsed):
        return parsed
    else:
        print(f"⚠️ Unexpected format, skipping: {type(parsed)}")
        return []  # ← return empty list so extend in build_qa_dataset is safe


def build_qa_dataset():
    all_qa = []

    for txt_file in sorted(CLEANED_DIR.glob("*.txt")):
        text = txt_file.read_text(encoding="utf-8")
        section_name = txt_file.stem.replace("_", " ")
        print(f"Generating Q&A for: {section_name}")

        try:
            pairs = generate_qa_from_chunk(text, section_name)
            all_qa.extend(pairs)
            print(f"  ✅ {len(pairs)} pairs generated")
        except Exception as e:
            print(f"  ⚠️  Skipped {txt_file.name}: {e}")

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_FILE.write_text(json.dumps(all_qa, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n✅ Saved {len(all_qa)} Q&A pairs to {OUTPUT_FILE}")


if __name__ == "__main__":
    build_qa_dataset()

Generating Q&A for: adhd differential reference
  ✅ 3 pairs generated
Generating Q&A for: adhd functional consequences
  ✅ 3 pairs generated
Generating Q&A for: adhd overview
⚠️ Unexpected format, skipping: <class 'str'>
  ✅ 0 pairs generated
Generating Q&A for: adhd symptoms reference
  ✅ 3 pairs generated

✅ Saved 9 Q&A pairs to validate\q_a.json


In [21]:
import json
from deepeval import evaluate
from deepeval.metrics import GEval, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCaseParams
from evaluate_rag import create_deep_eval_test_cases

# Set llm model for evaluation of the question and answers 
LLM_MODEL = "gpt-4o"

# Define evaluation metrics
correctness_metric = GEval(
    name="Correctness",
    model=LLM_MODEL,
    evaluation_params=[
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Determine whether the actual output is factually correct based on the expected output."
    ],
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.8,
    model=LLM_MODEL,
    include_reason=False
)

relevance_metric = ContextualRelevancyMetric(
    threshold=.8,
    model=LLM_MODEL,
    include_reason=True
)

def evaluate_rag(query_engine, num_questions: int = 5) -> None:
    """
    Evaluate the RAG system using predefined metrics.

    Args:
        query_engine: Query engine to ask questions and get answers along with retrieved context.
        num_questions (int): Number of questions to evaluate (default: 5).
    """


    # Load questions and answers from JSON file
    q_a_file_name = "./validate/q_a.json"
    with open(q_a_file_name, "r", encoding="utf-8") as json_file:
        q_a = json.load(json_file)

    questions = [qa["question"] for qa in q_a][:num_questions]
    ground_truth_answers = [qa["answer"] for qa in q_a][:num_questions]
    generated_answers = []
    retrieved_documents = []

    # Generate answers and retrieve documents for each question
    for question in questions:
        response = query_engine.query(question)
        context = [doc.text for doc in response.source_nodes]
        retrieved_documents.append(context)
        generated_answers.append(response.response)

    # Create test cases and evaluate
    test_cases = create_deep_eval_test_cases(questions, ground_truth_answers, generated_answers, retrieved_documents)
    evaluate(
        test_cases=test_cases,
        metrics=[correctness_metric, faithfulness_metric, relevance_metric]
    )

In [22]:
query_engine  = vector_store_index.as_query_engine(similarity_top_k=2)
evaluate_rag(query_engine, num_questions=1)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4o, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.8297978182738535, threshold: 0.5, strict: False, evaluation model: gpt-4o, reason: The actual output correctly identifies the primary symptoms of ADHD as inattention and hyperactivity-impulsivity, and oppositional defiant disorder as defiance and irritability. However, it does not explicitly mention the aversion to demanding tasks and forgetting instructions for ADHD, which are part of the expected output. The explanation of the disorders is mostly aligned with the expected output but lacks some specific details., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.8, strict: False, evaluation model: gpt-4o, reason: None, error: None)
  - ✅ Contextual Relevancy (score: 0.875, threshold: 0.8, strict: False, evaluation model: gpt-4o, reason: The score is 0.88 because while the context provides detailed symptoms of ADHD, it does not directly address the differentiation from oppositional defiant disorder. However, the statement '

⚠ WARNING: No hyperparameters logged.
» ]8;id=172621;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.24s | token cost: 0.019735000000000003 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Test Case Summary
 this single test case already tells a useful story — faithfulness is perfect meaning RAG is keeping the model grounded, correctness is high but not perfect meaning some nuance is still lost, and retrieval is good but not flawless. That's a realistic and credible finding.